# 02 - Node And Graph

This notebook introduces the smallest useful Neat image pipeline: an input node, an output node, and a graph connecting them. It also shows the basic ways to execute a graph: `graph.run(...)`, `graph.build(...)`, and `push(...)` / `pull(...)`.


In [1]:
from pathlib import Path
import numpy as np
import cv2
import pyneat


In [2]:
IMAGE_PATH = Path("../assets/images/image.png")
image_bgr = cv2.imread(str(IMAGE_PATH), cv2.IMREAD_COLOR)
print("image:", IMAGE_PATH, image_bgr.shape, image_bgr.dtype)


image: ../assets/images/image.png (563, 1000, 3) uint8


## Convert Image To A Tensor

OpenCV loads images as BGR. This graph will declare an RGB input, so convert BGR to RGB before creating the tensor.

In [3]:
image_rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)

tensor = pyneat.Tensor.from_numpy(
    image_rgb,
    copy=True,
    image_format=pyneat.PixelFormat.RGB,
)
print("tensor shape:", tuple(tensor.shape))


tensor shape: (563, 1000, 3)


**copy=True** means Neat should make its own copy of the NumPy array data.
So after this call, tensor owns an independent copy of image_rgb. If you later modify image_rgb, the Neat tensor should not change.

**image_format=pyneat.PixelFormat.RGB** tells Neat how to interpret the image channels.

## Create A Graph

`InputOptions` describes the image shape and format at graph ingress. This first graph is a pass-through: input node to output node.

In [4]:
height, width = image_rgb.shape[:2]
input_options = pyneat.InputOptions()
input_options.format = pyneat.Format.RGB
input_options.width = width
input_options.height = height
input_options.depth = 3

graph = pyneat.Graph("graph_name")
graph.add(pyneat.nodes.input(input_options))
graph.add(pyneat.nodes.output())


#### Describe The Graph

`graph.describe()` is useful while learning because it prints the graph structure Neat sees.

In [5]:
print(graph.describe())


0) Input  [mysrc]
1) Output


## Execute With `graph.run(...)`

`graph.run(...)` is the simplest synchronous path. Use it for one image, quick tests, and beginner examples.

In [6]:
outputs = graph.run([tensor])
out = outputs[0].to_numpy(copy=True)

print("output shape:", out.shape)
print("same pixels:", bool(np.array_equal(image_rgb, out)))


[1/4] Initializing runtime...
[2/4] Preparing input stream...
[3/4] Building graph...


output shape: (563, 1000, 3)


[4/4] Graph ready (6 ms)


same pixels: True


## Execute With `graph.build(...)` Then `run.run(...)`

`graph.build(...)` creates a reusable runtime object. `run.run(...)` is still synchronous, but the graph runtime is already built and can be reused.

In [7]:
run = graph.build([tensor])
try:
    outputs = run.run([tensor], timeout_ms=1000)
    out = outputs[0].to_numpy(copy=True)
    print("runner output shape:", out.shape)
    print("same pixels:", bool(np.array_equal(image_rgb, out)))
finally:
    run.close()


runner output shape: (563, 1000, 3)
same pixels: True


[1/4] Initializing runtime...
[2/4] Preparing input stream...
[3/4] Building graph...
[4/4] Graph ready (3 ms)


## Execute With `push(...)` And `pull(...)`

`push(...)` sends data into a built graph. `pull(...)` receives data from it. This is the pattern behind video, RTSP, and async pipelines.

**Note :** Here we are use node name **`image`** and **`out`** in graph and the same is used to push the input data and pull the output data

In [8]:
stream_graph = pyneat.Graph("named_image_passthrough")
stream_graph.add(pyneat.nodes.input("image"))
stream_graph.add(pyneat.nodes.output("out"))
stream_graph.connect("image", "out")

stream_run = stream_graph.build()
try:
    if not stream_run.push("image", [tensor]):
        raise RuntimeError(f"push failed: {stream_run.last_error()}")
    output = stream_run.pull("out", 2000)
finally:
    stream_run.close()

out_tensor = output.tensors[0]
out = out_tensor.to_numpy(copy=True)

print("same pixels:", bool(np.array_equal(image_rgb, out)))


[1/4] Initializing runtime...
[2/4] Building graph...
[3/4] Starting pipeline...
[4/4] Graph ready (0 ms)
Pipeline:
appsrc name=mysrc_2 is-live=true format=time do-timestamp=true block=true stream-type=stream max-bytes=0 caps="video/x-raw,format=RGB,width=1000,height=563,depth=3" ! appsink name=mysink_2 emit-signals=false sync=false max-buffers=4 drop=false enable-last-sample=false


same pixels: True


**NOTE-**

Because here we created named nodes:
```
    stream_graph.add(pyneat.nodes.input("image"))
    stream_graph.add(pyneat.nodes.output("out"))
```
Neat now knows there is an input endpoint called **`image`** and an output endpoint called **`out`**, 
but it does not automatically know that **`image`** should feed **`out`**.

So this line wires the data path:
```
    stream_graph.connect("image", "out")
```
Without it, the graph may have two endpoints but no actual edge between them:

## Sample as input to Graph

For RTSP streams we might want to add meta data as well like `timestamp` and `frame-id`, `steam-id`. In this case we can use `Sample` which attach these metadata with frame/tensor.

**Tensor**

    Good when you only need image/tensor data.

**Sample**

    Good when you also want metadata:
    stream_id
    frame_id
    pts_ns
    timestamps

In [9]:
sample = pyneat.Sample()
sample.kind = pyneat.SampleKind.Tensor
sample.tensor = tensor
sample.stream_id = "tutorial"
sample.frame_id = 1
sample.pts_ns = 0

stream_graph = pyneat.Graph("named_image_passthrough")
stream_graph.add(pyneat.nodes.input("image"))
stream_graph.add(pyneat.nodes.output("out"))
stream_graph.connect("image", "out")

stream_run = stream_graph.build()
try:
    if not stream_run.push("image", [sample]):
        raise RuntimeError(f"push failed: {stream_run.last_error()}")
    output = stream_run.pull("out", 2000)
finally:
    stream_run.close()

out_tensor = output.tensors[0]
out = out_tensor.to_numpy(copy=True)

print("pulled kind:", output.kind)
print("pulled stream:", output.stream_id, "frame:", output.frame_id)
print("same pixels:", bool(np.array_equal(image_rgb, out)))



pulled kind: SampleKind.TensorSet
pulled stream: tutorial frame: 1
same pixels: True


[1/4] Initializing runtime...
[2/4] Building graph...
[3/4] Starting pipeline...
[4/4] Graph ready (0 ms)


## What To Remember

`graph.run(...)` is easiest for one-shot synchronous execution.

`graph.build(...)` creates a reusable runtime.

`run.run(...)` is synchronous execution through that runtime.

`push(...)` and `pull(...)` separate input from output, which is what real-time pipelines need.

## References

- Public tutorial collection: [core/tutorials](https://github.com/sima-neat/core/tree/main/tutorials).
- Build and run a graph: [build_inference_pipeline.py](https://github.com/sima-neat/core/blob/main/tutorials/004_build_inference_pipeline/build_inference_pipeline.py).
- Named input/output push-pull: [build_a_custom_data_graph.py](https://github.com/sima-neat/core/blob/main/tutorials/013_build_a_custom_data_graph/build_a_custom_data_graph.py).